# Playground — Manual SetFit lab

2-class sandbox on **ArcMMLU + MedMCQA + MMLU_management** (swap `source_preset` in CFG).

Local: `pip install -e ".[playground]"` then run cells.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "routers").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from routers.playground.manual_lab import DEFAULT_CFG, build_train_eval, eval_with_report, generate_for_class, train_setfit
from routers.synthetic.ollama_server import ensure_ollama_server

ensure_ollama_server(wait_seconds=60)
print("ROOT:", ROOT)

In [ ]:
CFG = {
    **DEFAULT_CFG,
    "source_preset": "arc_med_mmlu",
    "domains": None,
    "train_per_class": 50,
    "eval_per_class": 100,
}

train_rows, eval_rows, domain_labels = build_train_eval(CFG)
print("domains:", domain_labels)
print("train:", len(train_rows), "eval:", len(eval_rows))

In [ ]:
model_base, id2label = train_setfit(train_rows, CFG)
metrics_base = eval_with_report(model_base, id2label, eval_rows)
print("baseline macro_f1:", metrics_base["macro_f1"])

In [ ]:
synth_rows = []
for label in domain_labels:
    refs = [r for r in train_rows if r["gold"] == label][:5]
    synth_rows.extend(generate_for_class(refs, label, CFG, domain_labels))
print("synthetic:", len(synth_rows))

In [ ]:
train_aug = train_rows + synth_rows
model_boost, id2label2 = train_setfit(train_aug, CFG)
metrics_boost = eval_with_report(model_boost, id2label2, eval_rows)
print("boost macro_f1:", metrics_boost["macro_f1"])